# Nexo Lab — Proyecto Significativo 3
**Lorena Abigail Coca Ccorpa · SENATI**

Semanas 9 a 12: datos, desarrollo web y visión por computadora. Ejecutar desde la raíz del repositorio o desde esta carpeta. No se necesitan claves ni servicios externos.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd() if (Path.cwd()/"app.py").exists() else Path.cwd().parent
sys.path.insert(0, str(root))
from app import app, regression_data, regression_model, digit_model
import numpy as np


## 1. Conocer los datos
240 filas sintéticas, semilla 42. No representan cotizaciones reales. Variables: gramos, horas, complejidad y costo.

In [ ]:
x, y = regression_data()
datos = np.column_stack((x,y))
print("Columnas: gramos, horas, complejidad, costo")
print(datos[:5].round(2))
print("Promedios:", datos.mean(axis=0).round(2))
print("Desviaciones:", datos.std(axis=0).round(2))

## 2. Entrenar y evaluar
Separación reproducible: 180 filas para entrenamiento y 60 para prueba. El modelo se ajusta solo con entrenamiento. MAE es error absoluto medio; R² describe variación explicada sobre prueba.

In [ ]:
modelo, metricas = regression_model()
print({k:v for k,v in metricas.items() if k != "points"})
print("Estimación para 150 g, 4 h, complejidad 40:", modelo.predict([[150,4,40]])[0])

## 3. Explicar una predicción
La regresión suma un costo base y los aportes de cada variable. La calidad de la predicción depende de la representatividad de los datos.

In [ ]:
valores = np.array([150,4,40])
print("Costo base:", modelo.intercept_)
print("Aportes [material, tiempo, complejidad]:", valores * modelo.coef_)
print("Total:", modelo.intercept_ + np.dot(valores, modelo.coef_))

## 4. Clasificación de imágenes
Digits contiene 1797 imágenes de 8×8. El pipeline normaliza los trazos, estandariza características y ajusta regresión logística. No es una red neuronal. Evaluación estratificada: 1347/450.

In [ ]:
clasificador, evaluacion = digit_model()
print("Exactitud:", evaluacion["accuracy"])
print(np.array(evaluacion["confusion"]))

## 5. Experimentar con OpenCV
El backend procesa imágenes en memoria. Compara cambios en luminosidad, contraste, nitidez y bordes. La detección de rostros no identifica personas.

In [ ]:
import cv2
from IPython.display import display, Image
imagen = cv2.imread(str(root/"static"/"vision-sample.jpg"))
imagen = cv2.resize(imagen, (600,400))
gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
bordes = cv2.Canny(cv2.GaussianBlur(gris,(5,5),0),50,125)
display(Image(data=cv2.imencode(".png",bordes)[1].tobytes()))

## 6. Del cuaderno a la API
La misma lógica alimenta el aplicativo Flask y las pruebas. Git registra los cambios; Render sirve el frontend y el backend.

In [ ]:
cliente = app.test_client()
print(cliente.get("/health").json)
print(cliente.post("/api/predict",json={"grams":150,"hours":4,"complexity":40}).json)

## Preguntas para el informe
1. ¿Por qué separar entrenamiento y prueba?
2. ¿Qué pasaría si los datos de costos reales no siguieran una relación lineal?
3. ¿Por qué la confianza de una predicción no garantiza un acierto?
4. ¿Qué cambia al aumentar el umbral de Canny?
5. ¿Qué aporta Git al trabajo y cómo se conecta con el despliegue?

**Herramientas exploradas en clase:** PyTorch y Teachable Machine. **Herramientas implementadas aquí:** OpenCV y scikit-learn. No se atribuyen resultados a modelos que no se ejecutan.